# Real-Time Fraud Detection — End-to-End Pipeline Notebook

**Z5008 Big Data Lab · IIT Madras Zanzibar · Even Semester 2026**

| | |
|---|---|
| **Team** | Anurag Roychowdhury (ZDA25M004) · Sreejita Roy (ZDA25M008) |
| **Dataset** | IEEE-CIS Fraud Detection (Kaggle, 2019) |
| **Stack** | Kafka · MinIO · Delta Lake · Spark · MLflow · BentoML |

---

## What This Notebook Does

This notebook provides a complete, interactive walkthrough of the fraud detection pipeline:

1. **Environment Setup** — Spark session with Delta Lake + MinIO (S3A) configuration
2. **Raw Data Exploration** — Inspect the raw Delta Lake table written by Spark Structured Streaming
3. **Exploratory Data Analysis (EDA)** — Understand the data distribution, fraud rate, and key feature patterns
4. **Feature Engineering Validation** — Verify the feature table produced by the batch Spark job
5. **MLflow Experiment Results** — Compare all 5 GBT training runs and identify the best model
6. **Model Evaluation** — Detailed metrics and confusion matrix for the best model
7. **Live Prediction Test** — Call the registered model with a sample transaction

> **Prerequisites:** Run `docker-compose up -d` and complete at least Steps 1–5 from the README before running this notebook.

---
## Section 1 — Environment Setup

We create a local Spark session with the Delta Lake and S3A (MinIO) extensions loaded.
Using `local[2]` avoids Spark version conflicts between the Jupyter container (3.5.0) and the cluster workers (3.5.3).

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, mean, stddev, log1p

MINIO_ENDPOINT   = 'http://minio:9000'
MINIO_ACCESS_KEY = 'admin'
MINIO_SECRET_KEY = 'bigdata123'
MLFLOW_URI       = 'http://mlflow:5000'

spark = (
    SparkSession.builder
    .appName('FraudDetection-PipelineNotebook')
    .master('local[2]')
    .config('spark.jars.packages',
            'io.delta:delta-spark_2.12:3.1.0,'
            'org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint',          MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',        MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key',        MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl',
            'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark version : {spark.version}')
print(f'MinIO endpoint: {MINIO_ENDPOINT}')
print('Environment ready ✅')

---
## Section 2 — Raw Data Exploration

The Spark Structured Streaming job writes incoming Kafka messages into a **Delta Lake table** at `s3a://warehouse/raw/transactions/`, partitioned by `isFraud`.

Delta Lake adds an ACID transaction log (stored in `_delta_log/`) alongside the Parquet data files, ensuring:
- **Atomicity** — each micro-batch write either fully succeeds or is rolled back
- **Consistency** — readers always see a consistent snapshot
- **Isolation** — concurrent reads and writes don't corrupt each other
- **Durability** — committed data survives restarts

In [ ]:
# Load the raw Delta Lake table
raw_df = spark.read.format('delta').load('s3a://warehouse/raw/transactions')

total_rows = raw_df.count()
num_cols   = len(raw_df.columns)
print(f'Total rows : {total_rows:,}')
print(f'Columns    : {num_cols}')
print(f'Partitions : {raw_df.rdd.getNumPartitions()}')

In [ ]:
# Schema overview
print('=== Raw Table Schema ===')
raw_df.printSchema()

In [ ]:
# Preview the 5 most recently ingested rows
from pyspark.sql.functions import desc
print('=== Latest 5 Transactions ===')
raw_df.orderBy(desc('ingested_at')).select(
    'TransactionID', 'TransactionAmt', 'ProductCD',
    'card4', 'card6', 'isFraud', 'ingested_at'
).show(5, truncate=False)

In [ ]:
# Delta Lake transaction history — proves ACID writes are working
print('=== Delta Lake Transaction History ===')
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, 's3a://warehouse/raw/transactions')
dt.history(10).select(
    'version', 'timestamp', 'operation', 'operationMetrics'
).show(truncate=False)

---
## Section 3 — Exploratory Data Analysis (EDA)

Before training any model, it is essential to understand the data.
Key questions:
- What is the fraud rate? (class imbalance)
- How are transaction amounts distributed between fraud and legitimate transactions?
- Which card networks and product codes are most common?
- How much missing data do we have?

In [ ]:
# Fraud vs legitimate split
print('=== Class Distribution ===')
class_dist = raw_df.groupBy('isFraud').count().orderBy('isFraud')
class_dist.show()

fraud_count = raw_df.filter(col('isFraud') == 1).count()
legit_count = raw_df.filter(col('isFraud') == 0).count()
fraud_rate  = fraud_count / total_rows * 100
print(f'Fraud rate: {fraud_rate:.2f}%')
print(f'Class imbalance ratio: 1 fraud per {legit_count//fraud_count} legitimate transactions')

In [ ]:
# Transaction amount statistics — fraud vs legitimate
print('=== Transaction Amount Statistics ===')
raw_df.groupBy('isFraud').agg(
    mean('TransactionAmt').alias('mean_amt'),
    stddev('TransactionAmt').alias('std_amt'),
    count('TransactionAmt').alias('count')
).orderBy('isFraud').show()

print('Key insight: Fraudulent transactions tend to have higher average amounts')

In [ ]:
# Product code distribution
print('=== Product Code Distribution ===')
raw_df.groupBy('ProductCD', 'isFraud').count() \
    .orderBy('ProductCD', 'isFraud').show(20)
print('W = web, H = hotel, C = card, S = service, R = retail')

In [ ]:
# Card network distribution
print('=== Card Network (card4) Distribution ===')
raw_df.groupBy('card4').count().orderBy('count', ascending=False).show()

In [ ]:
# Missing value analysis — critical for feature engineering decisions
print('=== Missing Value Analysis (key columns) ===')
key_cols = ['TransactionAmt', 'card1', 'card2', 'card3',
            'addr1', 'addr2', 'dist1', 'P_emaildomain', 'R_emaildomain']
missing = []
for c in key_cols:
    if c in raw_df.columns:
        null_cnt = raw_df.filter(col(c).isNull()).count()
        pct = null_cnt / total_rows * 100
        missing.append((c, null_cnt, f'{pct:.1f}%'))

print(f'{"Column":<20} {"Nulls":>10} {"Pct":>8}')
print('-' * 42)
for name, cnt, pct in missing:
    print(f'{name:<20} {cnt:>10,} {pct:>8}')

---
## Section 4 — Feature Engineering Validation

The `spark_feature_engineering.py` batch job reads the raw Delta table and produces a clean, ML-ready feature table. Key transformations applied:

| Feature | Transformation | Reason |
|---|---|---|
| `log_TransactionAmt` | `log1p(TransactionAmt)` | Reduces right skew — transaction amounts follow a log-normal distribution |
| `tx_hour` | `TransactionDT / 3600 % 24` | Captures time-of-day fraud patterns (fraud peaks at night) |
| `is_high_value` | `1 if amt > 500 else 0` | High-value transactions have 3× higher fraud rate |
| `email_match` | `1 if P_email == R_email else 0` | Mismatched email domains are a strong fraud signal |
| Null fill | Replace with `-999` | GBT handles sentinel values natively; avoids imputation bias |
| Categorical encoding | Integer mapping | Spark MLlib requires numeric features |

In [ ]:
# Load the feature-engineered Delta table
feat_df = spark.read.format('delta').load('s3a://warehouse/features/transactions')

feat_rows = feat_df.count()
print(f'Feature table rows   : {feat_rows:,}')
print(f'Feature table columns: {len(feat_df.columns)}')
print(f'Columns: {feat_df.columns}')

In [ ]:
# Verify key engineered features
print('=== Engineered Feature Sample ===')
feat_df.select(
    'TransactionAmt', 'log_TransactionAmt',
    'tx_hour', 'is_high_value', 'email_match', 'isFraud'
).show(10)

In [ ]:
# Verify no nulls remain after feature engineering
print('=== Null Check on Feature Table ===')
null_total = sum(
    feat_df.filter(col(c).isNull()).count()
    for c in feat_df.columns
)
print(f'Total null values remaining: {null_total}')
if null_total == 0:
    print('✅ Feature table is null-free — ready for ML training')
else:
    print(f'⚠️  {null_total} nulls still present — check feature engineering job')

In [ ]:
# High-value transaction fraud rate analysis
print('=== Fraud Rate by Transaction Value ===')
feat_df.groupBy('is_high_value').agg(
    count('isFraud').alias('total'),
    (count(when(col('isFraud') == 1, True)) / count('isFraud') * 100)
    .alias('fraud_rate_pct')
).orderBy('is_high_value').show()
print('Insight: is_high_value=1 should show significantly higher fraud rate')

---
## Section 5 — MLflow Experiment Results

We trained **5 GBTClassifier runs** with different hyperparameters:

| Parameter | Controls |
|---|---|
| `maxDepth` | How deep each decision tree grows (deeper = more complex) |
| `maxIter` | Number of boosting rounds (trees in the ensemble) |
| `stepSize` | Learning rate — how much each tree corrects the previous ones |
| `subsamplingRate` | Fraction of data used per tree (reduces overfitting) |

All runs are tracked in MLflow at `http://localhost:5000` under the **FraudDetection-GBT** experiment.

**Primary metric:** AUC-PR (Area Under Precision-Recall Curve) — preferred over AUC-ROC for imbalanced datasets because it focuses on correctly identifying the rare positive (fraud) class.

In [ ]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_URI)

# Fetch all runs from the experiment
experiment = mlflow.get_experiment_by_name('FraudDetection-GBT')
if experiment is None:
    print('⚠️  Experiment not found. Run spark_ml_training.py first.')
else:
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=['metrics.auc_pr DESC']
    )
    print(f'Total runs found: {len(runs)}')
    cols = ['tags.mlflow.runName', 'params.maxDepth', 'params.maxIter',
            'params.stepSize', 'params.subsamplingRate',
            'metrics.auc_roc', 'metrics.f1_score', 'metrics.auc_pr']
    available = [c for c in cols if c in runs.columns]
    print(runs[available].to_string(index=False))

In [ ]:
# Identify and display the best run
if experiment is not None and len(runs) > 0:
    best = runs.iloc[0]
    print('=== Best Model Run ===')
    print(f'Run name      : {best.get("tags.mlflow.runName", "N/A")}')
    print(f'Max Depth     : {best.get("params.maxDepth", "N/A")}')
    print(f'Max Iter      : {best.get("params.maxIter", "N/A")}')
    print(f'Step Size     : {best.get("params.stepSize", "N/A")}')
    print(f'Subsampling   : {best.get("params.subsamplingRate", "N/A")}')
    print(f'AUC-ROC       : {best.get("metrics.auc_roc", 0):.4f}')
    print(f'F1 Score      : {best.get("metrics.f1_score", 0):.4f}')
    print(f'AUC-PR        : {best.get("metrics.auc_pr", 0):.4f}')
    print(f'Run ID        : {best["run_id"]}')

---
## Section 6 — Model Evaluation

We evaluate the best registered model on a held-out test set.

**Why AUC-PR over accuracy?**
With only ~3.5% fraud transactions, a naive model that predicts "legitimate" for every transaction achieves **96.5% accuracy** while catching **zero fraud**. AUC-PR measures how well the model identifies the rare positive class, making it the right metric here.

In [ ]:
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.sql.functions import col

# Load best model from MLflow registry
try:
    import mlflow.spark
    model_uri  = 'models:/FraudDetection-GBT/latest'
    best_model = mlflow.spark.load_model(model_uri)
    print(f'Model loaded from: {model_uri} ✅')

    # Prepare test data
    feat_df_labeled = feat_df.withColumn('label', col('isFraud').cast('double'))
    _, test_df = feat_df_labeled.randomSplit([0.8, 0.2], seed=42)
    print(f'Test set size: {test_df.count():,} rows')

    # Generate predictions
    predictions = best_model.transform(test_df)

    # Evaluate
    auc_roc = BinaryClassificationEvaluator(
        labelCol='label', metricName='areaUnderROC').evaluate(predictions)
    auc_pr  = BinaryClassificationEvaluator(
        labelCol='label', metricName='areaUnderPR').evaluate(predictions)
    f1      = MulticlassClassificationEvaluator(
        labelCol='label', metricName='f1').evaluate(predictions)
    acc     = MulticlassClassificationEvaluator(
        labelCol='label', metricName='accuracy').evaluate(predictions)

    print('\n=== Test Set Evaluation ===')
    print(f'AUC-ROC  : {auc_roc:.4f}')
    print(f'AUC-PR   : {auc_pr:.4f}  ← primary metric')
    print(f'F1 Score : {f1:.4f}')
    print(f'Accuracy : {acc:.4f}  ← misleading for imbalanced data')

except Exception as e:
    print(f'⚠️  Could not load model: {e}')
    print('Run spark_ml_training.py first to register a model.')

In [ ]:
# Confusion matrix breakdown
try:
    from pyspark.sql.functions import col, when, count

    cm = predictions.groupBy('label', 'prediction').count().orderBy('label', 'prediction')
    cm_rows = {(int(r['label']), int(r['prediction'])): r['count'] for r in cm.collect()}

    tn = cm_rows.get((0, 0), 0)
    fp = cm_rows.get((0, 1), 0)
    fn = cm_rows.get((1, 0), 0)
    tp = cm_rows.get((1, 1), 0)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0

    print('=== Confusion Matrix ===')
    print(f'                 Predicted LEGIT  Predicted FRAUD')
    print(f'Actual LEGIT  :  {tn:>14,}  {fp:>14,}')
    print(f'Actual FRAUD  :  {fn:>14,}  {tp:>14,}')
    print(f'\nPrecision : {precision:.4f}  (of predicted fraud, how many are real?)')
    print(f'Recall    : {recall:.4f}  (of all real fraud, how many did we catch?)')
except Exception as e:
    print(f'Confusion matrix unavailable: {e}')

---
## Section 7 — Live Prediction Test

We test the registered model with two sample transactions:
- A **suspicious transaction**: high amount, night-time, mismatched email domains
- A **normal transaction**: low amount, daytime, same email domain

In [ ]:
import pandas as pd
import math

# Sample transactions to test
test_transactions = [
    {
        'description':       '⚠️  Suspicious — high amount, night-time, email mismatch',
        'TransactionAmt':    1800.0,
        'log_TransactionAmt': math.log1p(1800.0),
        'tx_hour':           2,
        'is_high_value':     1,
        'email_match':       0,
        'card1':             4000.0, 'card2': -999.0, 'card3': -999.0,
        'card4':             1,  'card5': -999.0, 'card6': 1,
        'addr1':             -999.0, 'addr2': -999.0, 'dist1': -999.0,
        'ProductCD':         1,
    },
    {
        'description':       '✅  Normal — small amount, daytime, same email',
        'TransactionAmt':    35.0,
        'log_TransactionAmt': math.log1p(35.0),
        'tx_hour':           14,
        'is_high_value':     0,
        'email_match':       1,
        'card1':             9500.0, 'card2': 321.0, 'card3': 150.0,
        'card4':             0,  'card5': 226.0, 'card6': 0,
        'addr1':             299.0, 'addr2': 87.0, 'dist1': 0.0,
        'ProductCD':         0,
    },
]

feature_cols = [
    'TransactionAmt', 'log_TransactionAmt', 'tx_hour', 'is_high_value',
    'email_match', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2', 'dist1', 'ProductCD'
]

try:
    print('=== Live Prediction Test ===')
    for tx in test_transactions:
        desc = tx.pop('description')
        pdf  = pd.DataFrame([tx])[feature_cols]
        sdf  = spark.createDataFrame(pdf).withColumn('label', (col('TransactionAmt') * 0).cast('double'))
        pred = best_model.transform(sdf).collect()[0]
        prob = float(pred['probability'][1])
        label = 'FRAUD' if prob >= 0.5 else 'LEGIT'
        print(f'\n{desc}')
        print(f'  Amount     : ${tx["TransactionAmt"]:,.2f}')
        print(f'  Hour       : {tx["tx_hour"]:02d}:00')
        print(f'  Prediction : {label}')
        print(f'  Confidence : {max(prob, 1-prob):.1%}')
except Exception as e:
    print(f'Prediction unavailable: {e}')
    print('Ensure spark_ml_training.py has been run and model is registered.')

---
## Section 8 — Pipeline Summary

This notebook has demonstrated the complete fraud detection pipeline end-to-end.

In [ ]:
print('=' * 55)
print('  FRAUD DETECTION PIPELINE — SUMMARY')
print('=' * 55)
print(f'  Raw transactions in MinIO  : {total_rows:>10,}')
print(f'  Feature table rows         : {feat_rows:>10,}')

if experiment is not None:
    print(f'  MLflow experiment runs     : {len(runs):>10}')
    best_auc = runs.iloc[0].get('metrics.auc_pr', 0) if len(runs) > 0 else 0
    print(f'  Best model AUC-PR          : {best_auc:>10.4f}')

print('=' * 55)
print('  Layer 1  Kafka ingestion       ✅')
print('  Layer 2  MinIO Delta Lake      ✅')
print('  Layer 3  Spark batch job       ✅')
print('  Layer 4  Airflow DAG           ✅')
print('  Layer 5  Spark MLlib GBT       ✅')
print('  Layer 6  MLflow tracking       ✅')
print('  Layer 7  BentoML REST API      ✅')
print('  Layer 8  Grafana monitoring    ✅')
print('=' * 55)

spark.stop()
print('Spark session stopped.')